# On-Chain Analytics — Polygon
Interactive walkthrough of wallet profiling, token inspection, Uniswap V3 pool analysis, and gas estimation.

## 1. Connect to Polygon

In [1]:
from config.rpc import get_polygon_w3
from config.wallets import MY_ADDRESS
from config.tokens import POLYGON

w3 = get_polygon_w3()
print(f"Connected: {w3.is_connected()}")
print(f"Chain ID: {w3.eth.chain_id}")
print(f"Latest block: {w3.eth.block_number:,}")

Connected: True
Chain ID: 137
Latest block: 83,128,393


## 2. Wallet Profile

In [2]:
from analytics.wallet import WalletAnalyzer

wa = WalletAnalyzer(w3)
profile = wa.profile(MY_ADDRESS)

print(f"Address: {MY_ADDRESS}")
print(f"Transactions: {profile['tx_count']}")
print(f"\nBalances:")
for token, balance in profile["balances"].items():
    print(f"  {token}: {balance:,.6f}")

Address: 0xCB81E826aE200e0dA09b8813A939611702A5A8C0
Transactions: 57

Balances:
  MATIC: 5.454672
  USDC: 21.333193


## 3. Token Info
Look up any ERC-20 by contract address.

In [3]:
from analytics.tokens import TokenAnalyzer

ta = TokenAnalyzer(w3)

for symbol, addr in POLYGON.items():
    if symbol == "MATIC":
        continue  # native token, no ERC-20 contract
    try:
        info = ta.get_info(addr)
        supply = info["total_supply"] / (10 ** info["decimals"])
        print(f"{info['symbol']:>8} | decimals: {info['decimals']} | supply: {supply:>20,.2f} | {addr[:16]}...")
    except Exception as e:
        print(f"{symbol:>8} | error: {e}")

    WPOL | decimals: 18 | supply:       217,661,834.97 | 0x0d500B1d8E8eF3...
   USDT0 | decimals: 6 | supply:       924,279,842.13 | 0xc2132D05D31c91...
    USDC | decimals: 6 | supply:       696,974,837.93 | 0x3c499c542cEF5E...
    WETH | decimals: 18 | supply:            94,769.55 | 0x7ceB23fD6bC0ad...
     DAI | decimals: 18 | supply:       626,274,690.29 | 0x8f3Cf7ad23Cd3C...


## 4. Uniswap V3 Pool Lookup
Find the best liquidity pool for a pair and read its state.

In [4]:
from analytics.pools import PoolAnalyzer

pa = PoolAnalyzer(w3)

# Find the best WMATIC/USDC pool across fee tiers
best = pa.find_best_pool(POLYGON["WMATIC"], POLYGON["USDC"])
print(f"Best pool: {best['address']}")
print(f"Fee tier: {best['fee_pct']}")
print(f"Liquidity: {best['liquidity']:,}")

Best pool: 0xB6e57ed85c4c9dbfEF2a68711e9d6f36c56e0FcB
Fee tier: 0.05%
Liquidity: 6,863,877,225,407,672,285


In [5]:
# Read full pool state (slot0, tokens, price)
state = pa.get_pool_state(best["address"])

# Adjust price for decimals (WMATIC=18, USDC=6)
matic_price = pa.get_pool_price_adjusted(best["address"], 18, 6)

print(f"Token0: {state['token0']}")
print(f"Token1: {state['token1']}")
print(f"Current tick: {state['tick']}")
print(f"\nMATIC price: ${matic_price:.4f}")

Token0: 0x0d500B1d8E8eF31E21C99d1Db9A6444d3ADf1270
Token1: 0x3c499c542cEF5E3811e1192ce70d8cC03d5c3359
Current tick: -298593

MATIC price: $0.1079


## 5. Scan All WMATIC Pairs

In [6]:
import pandas as pd

pairs = pa.scan_pairs()
df = pd.DataFrame(pairs).sort_values("liquidity", ascending=False)
df

,address,fee,fee_pct,liquidity,pair
4,0xFE530931dA161232Ec76A7c3bEA7D36cF3811A0d,3000,0.30%,31115330216310272373983,WMATIC/DAI
3,0x86f1d8390222A3691C28938eC7404A1661E618e0,500,0.05%,20540474027919364344642,WMATIC/WETH
1,0x9B08288C3Be4F62bbf8d1C20Ac9C5e6f9467d8B7,500,0.05%,7744222641076771580,WMATIC/USDT
2,0xB6e57ed85c4c9dbfEF2a68711e9d6f36c56e0FcB,500,0.05%,6863877225407672285,WMATIC/USDC
0,0xf835EE44988904830d2a9A48cC0f0318dAef2BAd,100,0.01%,0,WMATIC/MATIC


## 6. Gas Prices + USD Cost

In [7]:
from functions.gas_estimate import get_gas_price

for chain in ["polygon", "ethereum"]:
    price = get_gas_price(chain)
    usd = f"${price['transfer_cost_usd']:.6f}" if price["transfer_cost_usd"] else "N/A"
    native = "MATIC" if chain == "polygon" else "ETH"
    print(f"{chain.title():>10} | {float(price['gwei']):>10.2f} gwei | {native} price: ${price['native_usd']:.4f} | transfer cost: {usd}")

   Polygon |     684.34 gwei | MATIC price: $0.1079 | transfer cost: $0.001550
  Ethereum |       0.04 gwei | ETH price: $0.0000 | transfer cost: $0.000000


## 7. Token Transfer Events
Pull recent `Transfer` events for any token. Useful for watching whale moves, tracking flows.

In [8]:
# Scan last 100 blocks for USDC transfers on Polygon
latest = w3.eth.block_number
transfers = ta.get_transfer_events(POLYGON["USDC"], latest - 100)

print(f"Found {len(transfers)} USDC transfers in last 100 blocks\n")

df_transfers = pd.DataFrame(transfers)
if not df_transfers.empty:
    df_transfers = df_transfers.sort_values("value", ascending=False)
    print("Top 10 by value:")
    df_transfers.head(10)

Found 1984 USDC transfers in last 100 blocks

Top 10 by value:


In [11]:
df_transfers.head(10)

,from,to,value,block,tx_hash
1933,0x63242A4Ea82847b20E506b63B0e2e2eFF0CC6cB0,0x8Be46b25d59616e594f0A9e20147fb14c1B989d9,350121.659764,83128424,e4e25ae019d79ca7c304151d35bb8d01df6cefe176b635...
1931,0x7BCD99AE248E86EcaB7A5B23451D73825117F54D,0x63242A4Ea82847b20E506b63B0e2e2eFF0CC6cB0,280097.560024,83128424,e4e25ae019d79ca7c304151d35bb8d01df6cefe176b635...
1932,0x67366782805870060151383F4BbFF9daB53e5cD6,0x63242A4Ea82847b20E506b63B0e2e2eFF0CC6cB0,70024.099740,83128424,e4e25ae019d79ca7c304151d35bb8d01df6cefe176b635...
1250,0x7F3D3F7495E033D8Cd17BDeB08B70102572DB7a6,0x233c5370CCfb3cD7409d9A3fb98ab94dE94Cb4Cd,10672.868936,83128393,ec2e2813cf9070d0fdf11c6f712b421653a2c72c26e3c9...
1935,0xFc4075AA7dC27192b16BaF1AF2B09c40a7A88340,0x4E301851788f446F5E8eE64A618809Bd72dF2BBF,5000.000000,83128424,aa0f2650256f09a483cab3ab9d72102e4cb060aa5df922...
1769,0xFc4075AA7dC27192b16BaF1AF2B09c40a7A88340,0x4E301851788f446F5E8eE64A618809Bd72dF2BBF,5000.000000,83128414,8a93d10fdaabc68355a1582dd06429edbdb601f867162d...
1861,0xFc4075AA7dC27192b16BaF1AF2B09c40a7A88340,0x4E301851788f446F5E8eE64A618809Bd72dF2BBF,3005.000000,83128419,0b7a825a3018b34f5569e0503aa511b05337728dc9d3d1...
661,0x6D28c6A85C28D3e828c94a2efbBa744aC041D701,0x14Dc866f6622a0E2c0d864a1f543d8a96f7cB5c0,2416.202772,83128368,536a6900bd740ed68c0499b29be6bee05c7a806b03ee6b...
538,0x037ebD1b9b33096F95f338A7B3445ec24BC52C99,0x48922081c3d3255117f2Bb949f9Bc489A07E3abd,2088.642896,83128361,529e8c8b10a3148e590f077d6ce533108b7c68f1023501...
1355,0xf70da97812CB96acDF810712Aa562db8dfA3dbEF,0xF5042e6ffaC5a625D4E7848e0b01373D8eB9e222,2035.174421,83128398,473f52e585c3a0281512efc0438b4bff685ff5470fac8a...


## 8. Net Flow Analysis
Who's accumulating vs dumping over a block range?

In [12]:
holders = ta.top_holders_from_transfers(POLYGON["USDC"], latest - 100)

df_holders = pd.DataFrame(holders)
if not df_holders.empty:
    print(f"Top accumulators (net inflow) over last 100 blocks:\n")
    df_holders.head(10)

Top accumulators (net inflow) over last 100 blocks:



In [14]:
df_holders.head(10)

,address,net_flow
0,0x8Be46b25d59616e594f0A9e20147fb14c1B989d9,350121.659764
1,0x4E301851788f446F5E8eE64A618809Bd72dF2BBF,32986.850000
2,0x1AB4973a48dc892Cd9971ECE8e01DcC7688f8F23,16292.484066
3,0x233c5370CCfb3cD7409d9A3fb98ab94dE94Cb4Cd,10672.869240
4,0xd5283Cabd64fd553F58dA2Cc64ec9Ab5224c14a1,3252.661479
5,0xB33bD56d4192E8E4e6a02e93Eabc732165199346,2773.460492
6,0xcE5774Bf9f3656E1Dda669C11344c67ceA8109Af,2416.202772
7,0x48922081c3d3255117f2Bb949f9Bc489A07E3abd,2123.383932
8,0x4C97D88246F6A22a7cD383Ecca3ad19FA909b9D9,1998.580000
9,0x66DbFf2Ce099d19b4E8C5Dc8B254EC7aeAF5E642,1927.714160
